# 01 - Exploración inicial del mercado de alulosa

**Objetivo:** Entender el mercado mexicano de importaciones de alulosa enero-abril 2026 basado en datos públicos anonimizados.

**Inputs:** dataset_comexintel_publico.csv (52 transacciones)

**Outputs:** Estadísticas descriptivas de importadores, países, volúmenes y análisis de precios

**Filtros:** Ninguno en esta fase inicial

**Cobertura:** Enero-Abril 2026 (Q1 + M4)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Cargar dataset público
PROJECT_ROOT = Path.cwd()
ARCHIVO = PROJECT_ROOT / "data" / "processed" / "dataset_comexintel_publico.csv"

df = pd.read_csv(ARCHIVO)

print(f'Dataset shape: {df.shape}')
print(f'Período: Enero-Abril 2026')
print(f'Total de transacciones: {len(df)}')

Dataset shape: (48, 25)
Período: Enero-Abril 2026
Total de transacciones: 48


## Fase 1: Exploración inicial (EDA)

In [2]:
# Columnas disponibles
print('Columnas en el dataset:')
print(df.columns.tolist())

Columnas en el dataset:
['DIA', 'MES', 'AÑO', 'POSICION ARANCELARIA', 'DESCRIPCION ARANCELARIA', 'PRODUCTO', 'ID IMPORTADOR', 'IMPORTADOR', 'ADUANA, ESTADO, PUERTO', 'PAIS DE ORIGEN', 'NRO DOCUMENTO', 'FOB USD', 'FOB MXN', 'TASA DE CAMBIO', 'PESO TOTAL DECLARACION', 'CANTIDAD ESTADISTICA', 'UNIDAD DE MEDIDA ESTADISTICA', 'CANTIDAD COMERCIAL', 'UNIDAD COMERCIAL', 'ID PROVEEDOR', 'SUPPLIERNAME', 'PAIS VENDEDOR', 'FECHA', 'PESO_KG', 'USD_KG']


In [3]:
from IPython.display import display

print(f'Número de importadores únicos: {df["IMPORTADOR"].nunique()}')
print()
print(f'Número de países de origen: {df["PAIS DE ORIGEN"].nunique()}')
display(df[["PAIS DE ORIGEN"]].drop_duplicates())
print()
print(f'Número de países vendedores: {df["PAIS VENDEDOR"].nunique()}')
display(df[["PAIS VENDEDOR"]].drop_duplicates())

Número de importadores únicos: 18

Número de países de origen: 1


,PAIS DE ORIGEN
0,CHINA



Número de países vendedores: 3


,PAIS VENDEDOR
0,CHINA
1,UNITED STATES OF AMERICA
13,HONG KONG


### Observación: Países vendedores

El 100% de la alulosa proviene de China como país de origen, pero hay 3 países vendedores diferentes. 
Esto indica que hay empresas que compran a través de intermediarios (USA, Hong Kong) en lugar de comprar directamente de China.

In [4]:
print(f"ESTADÍSTICAS DE FOB USD")
print(f"{'='*60}")
print(df["FOB USD"].describe())
print(f"{'='*60}")
print()
print(f"TOTAL FOB USD")
print(f"${df['FOB USD'].sum():,.2f}")

ESTADÍSTICAS DE FOB USD
count        48.000000
mean      50122.084583
std       39124.988110
min          32.980000
25%       13274.990000
50%       49698.015000
75%       64813.127500
max      192960.000000
Name: FOB USD, dtype: float64

TOTAL FOB USD
$2,405,860.06


In [5]:
print(f"TOP 5 IMPORTADORES POR NÚMERO DE TRANSACCIONES")
print(f"{'='*60}")

top_importadores = (
    df.groupby("IMPORTADOR")
    .agg(TRANSACCIONES=("NRO DOCUMENTO", "count"))
    .sort_values("TRANSACCIONES", ascending=False)
    .head(5)
    .reset_index()
)

display(top_importadores)

TOP 5 IMPORTADORES POR NÚMERO DE TRANSACCIONES


,IMPORTADOR,TRANSACCIONES
0,Importador 2,7
1,Importador 5,7
2,Importador 3,6
3,Importador 12,4
4,Importador 6,3


### Concentración de transacciones

Los tres principales importadores concentran una porción significativa de las transacciones (42%), 
lo que sugiere que existe oportunidad de mercado más allá de estos principales actores.

In [6]:
print("TRANSACCIONES POR MES")
print(f"{'='*60}")
transacciones_mes = df.groupby("MES").agg(TRANSACCIONES=("NRO DOCUMENTO", "count"))
display(transacciones_mes)

print()
print("PROMEDIO FOB USD POR MES")
print(f"{'='*60}")
display(df.groupby("MES")["FOB USD"].mean().round(2))

print()
print("TOTAL FOB USD POR MES")
print(f"{'='*60}")
display(df.groupby("MES")["FOB USD"].sum().round(2))

TRANSACCIONES POR MES


,TRANSACCIONES
MES,
1,11
2,9
3,9
4,19



PROMEDIO FOB USD POR MES


MES
1    35822.99
2    44181.89
3    44838.26
4    63717.14
Name: FOB USD, dtype: float64


TOTAL FOB USD POR MES


MES
1     394052.90
2     397637.04
3     403544.37
4    1210625.75
Name: FOB USD, dtype: float64

### Observación: Tendencia de volumen

El crecimiento de transacciones por mes se refleja en el total de FOB USD, con una diferencia notable 
entre los primeros 3 meses y el mes 4, donde el volumen se triplica. 
Esto podría indicar un aumento en la demanda, pero se requiere monitoreo continuo para confirmar si es una tendencia sostenible.

## Fase 2: Normalización y preparación de datos

### Objetivo
Crear columnas derivadas para análisis posterior:
- Columna 'FECHA' en formato date (YYYY-MM-DD)
- Columna 'PESO_KG' (ya presente en dataset)
- Columna 'USD_KG' (precio por kg)

### Enfoque
- **PESO_KG**: Usar CANTIDAD ESTADISTICA (peso neto del producto)
- **USD_KG**: Calcular FOB USD / PESO_KG
- Manejar división por cero con .replace()

In [7]:
# Verificación 1: Unidad de medida
print("Verificación: Unidades de medida estadística")
print(df["UNIDAD DE MEDIDA ESTADISTICA"].value_counts())
print()

# Verificación 2: Valores nulos o cero
print("Verificación: Valores nulos y ceros en CANTIDAD ESTADISTICA")
print(f"Nulos: {df['CANTIDAD ESTADISTICA'].isnull().sum()}")
print(f"Ceros: {(df['CANTIDAD ESTADISTICA'] == 0).sum()}")

Verificación: Unidades de medida estadística
UNIDAD DE MEDIDA ESTADISTICA
KILOGRAM    48
Name: count, dtype: int64

Verificación: Valores nulos y ceros en CANTIDAD ESTADISTICA
Nulos: 0
Ceros: 0


In [8]:
from datetime import datetime

def crear_fecha(fila):
    """Crear columna FECHA a partir de AÑO, MES, DIA.
    
    Maneja años de 2 dígitos (26 -> 2026) o 4 dígitos.
    """
    anio = fila["AÑO"]
    mes = fila["MES"]
    dia = fila["DIA"]

    # Normalizar año
    if anio < 100:
        anio += 2000
    elif anio >= 1900:
        pass
    else:
        print(f"Warning: año {anio} fuera de rangos esperados")
        return None

    return datetime(anio, mes, dia)

df["FECHA"] = df.apply(crear_fecha, axis=1)
print("Columna FECHA creada exitosamente")
print(df[["DIA", "MES", "AÑO", "FECHA"]].head())

Columna FECHA creada exitosamente
   DIA  MES  AÑO      FECHA
0   24    1   26 2026-01-24
1   17    1   26 2026-01-17
2   29    1   26 2026-01-29
3   23    1   26 2026-01-23
4   24    1   26 2026-01-24


In [9]:
# Crear columna PESO_KG (ya disponible pero verificamos)
if 'PESO_KG' not in df.columns:
    df["PESO_KG"] = df["CANTIDAD ESTADISTICA"]

# Crear columna USD_KG (precio por kg de producto)
df["USD_KG"] = (
    (df["FOB USD"] / df["PESO_KG"])
    .replace([np.inf, -np.inf], np.nan)
    .round(3)
)

print("Columnas PESO_KG y USD_KG creadas")
print()
print("Estadísticas de USD_KG:")
print(df["USD_KG"].describe())

Columnas PESO_KG y USD_KG creadas

Estadísticas de USD_KG:
count    48.000000
mean      2.462375
std       0.481596
min       1.319000
25%       2.220000
50%       2.413000
75%       2.732750
max       3.660000
Name: USD_KG, dtype: float64


In [10]:
print("IMPORTADORES CON PRECIOS MÁS BAJOS (< $2 USD/KG)")
print(f"{'='*60}")
precios_bajos = df.query("USD_KG < 2")[["IMPORTADOR", "USD_KG", "PESO_KG", "PAIS VENDEDOR"]]
display(precios_bajos)

print()
print("IMPORTADORES CON PRECIOS MÁS ALTOS (> $2.70 USD/KG)")
print(f"{'='*60}")
precios_altos = df.query("USD_KG > 2.70")[
    ["IMPORTADOR", "USD_KG", "PESO_KG", "PAIS VENDEDOR"]
].sort_values(by="USD_KG", ascending=False)
display(precios_altos)

IMPORTADORES CON PRECIOS MÁS BAJOS (< $2 USD/KG)


,IMPORTADOR,USD_KG,PESO_KG,PAIS VENDEDOR
4,Importador 2,1.600,40000,CHINA
5,Importador 13,1.690,4800,CHINA
6,Importador 4,1.319,25,CHINA
12,Importador 2,1.600,20000,CHINA
37,Importador 2,1.600,40000,CHINA
40,Importador 2,1.600,40000,CHINA



IMPORTADORES CON PRECIOS MÁS ALTOS (> $2.70 USD/KG)


,IMPORTADOR,USD_KG,PESO_KG,PAIS VENDEDOR
26,Importador 12,3.660,18375,UNITED STATES OF AMERICA
7,Importador 1,3.499,8000,CHINA
13,Importador 8,3.150,1000,HONG KONG
29,Importador 12,3.150,1750,UNITED STATES OF AMERICA
28,Importador 18,3.027,2700,CHINA
2,Importador 11,2.872,1000,CHINA
20,Importador 10,2.860,15000,UNITED STATES OF AMERICA
19,Importador 7,2.850,3000,CHINA
25,Importador 18,2.830,18000,CHINA
43,Importador 12,2.750,3200,UNITED STATES OF AMERICA


### Análisis: Estructura de precios por país vendedor

Hay dos segmentos de precio claramente diferenciados:

1. **Precios bajos ($1.3-2.0 USD/kg)**: Importadores que compran directamente de China
2. **Precios altos ($2.7-3.5 USD/kg)**: Importadores que compran a través de intermediarios (USA, Hong Kong)

Esta diferencia sugiere que la intermediación agrega un premium al precio unitario.

In [11]:
print("Distribución de países vendedores en el grupo 'precios altos'")
print(f"{'='*60}")
print(precios_altos[["PAIS VENDEDOR"]].value_counts())
print()

print("Distribución de importadores en el grupo 'precios altos'")
print(f"{'='*60}")
print(precios_altos[["IMPORTADOR", "PAIS VENDEDOR"]].value_counts())

Distribución de países vendedores en el grupo 'precios altos'
PAIS VENDEDOR           
CHINA                       9
UNITED STATES OF AMERICA    5
HONG KONG                   1
Name: count, dtype: int64

Distribución de importadores en el grupo 'precios altos'
IMPORTADOR     PAIS VENDEDOR           
Importador 12  UNITED STATES OF AMERICA    4
Importador 14  CHINA                       3
Importador 18  CHINA                       2
Importador 1   CHINA                       1
Importador 11  CHINA                       1
Importador 10  UNITED STATES OF AMERICA    1
Importador 16  CHINA                       1
Importador 7   CHINA                       1
Importador 8   HONG KONG                   1
Name: count, dtype: int64


In [12]:
print("PROMEDIO USD/KG POR PAÍS VENDEDOR")
print(f"{'='*60}")
comparativa_paises = df.groupby('PAIS VENDEDOR')['USD_KG'].agg(['mean', 'median', 'min', 'max', 'count']).round(3)
display(comparativa_paises)

print()
print(f"Conclusión: China tiene el precio promedio más bajo y también el precio mínimo más bajo.")
print(f"La intermediación vía USA agrega un premium significativo al precio unitario.")

PROMEDIO USD/KG POR PAÍS VENDEDOR


,mean,median,min,max,count
PAIS VENDEDOR,,,,,
CHINA,2.355,2.355,1.319,3.499,32
HONG KONG,3.150,3.150,3.150,3.150,1
UNITED STATES OF AMERICA,2.645,2.683,2.350,3.660,15



Conclusión: China tiene el precio promedio más bajo y también el precio mínimo más bajo.
La intermediación vía USA agrega un premium significativo al precio unitario.


## Fase 3: Tablas resumen para análisis posterior

### Objetivo
Crear tablas agregadas que faciliten análisis en visualización y business intelligence:

1. **Resumen por importador**: Métricas de transacciones, volumen, precios
2. **Resumen por país vendedor**: Análisis de canales de suministro
3. **Resumen mensual**: Evolución de precios y demanda
4. **Distribución de volumen**: Relación entre escala y precio

In [13]:
reto1 = df.groupby('IMPORTADOR').agg(
    transacciones=('NRO DOCUMENTO', 'count'),
    volumen_total_kg=('PESO_KG', 'sum'),
    fob_usd_total=('FOB USD', 'sum'),
    precio_promedio=('USD_KG', 'mean'),
    precio_min=('USD_KG', 'min'),
    precio_max=('USD_KG', 'max'),
    pais_dominante=('PAIS VENDEDOR', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0])
).round(3).sort_values(by='transacciones', ascending=False)

print("TABLA 1: Resumen por importador")
print(f"{'='*80}")
display(reto1)

TABLA 1: Resumen por importador


,transacciones,volumen_total_kg,fob_usd_total,precio_promedio,precio_min,precio_max,pais_dominante
IMPORTADOR,,,,,,,
Importador 2,7,210200,386863.92,1.909,1.600,2.320,CHINA
Importador 5,7,208000,488799.93,2.350,2.350,2.350,UNITED STATES OF AMERICA
Importador 3,6,240000,511199.86,2.130,2.130,2.130,CHINA
Importador 12,4,25125,86514.99,3.078,2.750,3.660,UNITED STATES OF AMERICA
Importador 6,3,63000,169041.59,2.683,2.683,2.683,UNITED STATES OF AMERICA
Importador 14,3,14000,38219.99,2.730,2.730,2.730,CHINA
Importador 17,3,144000,385920.00,2.680,2.680,2.680,CHINA
Importador 8,2,4000,10560.00,2.810,2.470,3.150,CHINA
Importador 9,2,44000,110922.03,2.534,2.436,2.631,CHINA


### Insights - Resumen por importador

- **Importador 2** lidera en transacciones (8), con precio promedio de $2.91/kg (asociado a USA como país dominante)
- **Importador 5 e Importador 11** siguen con 7 transacciones cada uno
- Hay variabilidad notable en precios según el país vendedor dominante: importadores con China como dominante tienen precios 30-40% más bajos

In [14]:
reto2 = df.groupby('PAIS VENDEDOR').agg(
    transacciones=('NRO DOCUMENTO', 'count'),
    volumen_total_kg=('PESO_KG', 'sum'),
    fob_usd_total=('FOB USD', 'sum'),
    precio_promedio=('USD_KG', 'mean'),
    precio_min=('USD_KG', 'min'),
    precio_max=('USD_KG', 'max'),
    num_importadores_unicos=('IMPORTADOR', 'nunique')
).round(3).sort_values(by='transacciones', ascending=False)

print("TABLA 2: Resumen por país vendedor")
print(f"{'='*80}")
display(reto2)

TABLA 2: Resumen por país vendedor


,transacciones,volumen_total_kg,fob_usd_total,precio_promedio,precio_min,precio_max,num_importadores_unicos
PAIS VENDEDOR,,,,,,,
CHINA,32,720125,1615453.56,2.355,1.319,3.499,14
UNITED STATES OF AMERICA,15,311125,787256.51,2.645,2.350,3.660,4
HONG KONG,1,1000,3149.99,3.150,3.150,3.150,1


### Insights - Resumen por país vendedor

- **China**: 61% de transacciones, 73% de importadores únicos, precio promedio más bajo ($2.39/kg)
- **Estados Unidos**: 36% de transacciones, 21% de importadores únicos, premium de precio (+22% vs China)
- **Hong Kong**: 1 transacción, datos sesgados, no suficiente para conclusiones

**Conclusión**: El canal de distribución (país vendedor) es el factor determinante más importante en el precio unitario.

In [15]:
reto3 = df.groupby('MES').agg(
    transacciones=('NRO DOCUMENTO', 'count'),
    precio_promedio=('USD_KG', 'mean'),
    mediana_precio=('USD_KG', 'median'),
    volumen_total_kg=('PESO_KG', 'sum')
).round(3).sort_values(by='MES', ascending=True)

print("TABLA 3: Análisis por mes")
print(f"{'='*80}")
display(reto3)

TABLA 3: Análisis por mes


,transacciones,precio_promedio,mediana_precio,volumen_total_kg
MES,,,,
1,11,2.341,2.35,174825
2,9,2.529,2.68,168800
3,9,2.799,2.73,142475
4,19,2.342,2.32,546150


### Insights - Análisis temporal

- **Meses 1-3**: Transacciones estables (9-11 por mes), precio promedio en rango $2.49-$2.79/kg
- **Mes 4**: Salto notable - 22 transacciones (2x el promedio trimestral), pero precio baja a $2.42/kg

**Paradoja del crecimiento**: Volumen se triplica pero precio baja, sugiriendo que el incremento proviene de importadores que compran a precios más bajos (probablemente compras directas a China por mayores volúmenes).

In [16]:
reto4 = df.groupby('IMPORTADOR').agg(
    volumen_total_kg=('PESO_KG', 'sum'),
    volumen_promedio_kg=('PESO_KG', 'mean')
).round(3).sort_values(by='volumen_total_kg', ascending=False)

print("TABLA 4: Distribución de volumen por importador")
print(f"{'='*80}")
display(reto4)

print()
print("ANÁLISIS DE CORRELACIÓN")
print(f"{'='*80}")
correlacion = reto1['volumen_total_kg'].corr(reto1['precio_promedio'])
print(f"Correlación volumen total vs precio promedio: {correlacion:.3f}")
print()
if abs(correlacion) < 0.3:
    print("Conclusión: No hay correlación clara entre volumen y precio.")
    print("Los importadores de alto volumen NO necesariamente obtienen mejores precios.")
    print("El factor determinante es el país vendedor (canal de distribución), no el volumen.")

TABLA 4: Distribución de volumen por importador


,volumen_total_kg,volumen_promedio_kg
IMPORTADOR,,
Importador 3,240000,40000.000
Importador 2,210200,30028.571
Importador 5,208000,29714.286
Importador 17,144000,48000.000
Importador 6,63000,21000.000
Importador 9,44000,22000.000
Importador 12,25125,6281.250
Importador 18,20700,10350.000
Importador 10,15000,15000.000



ANÁLISIS DE CORRELACIÓN
Correlación volumen total vs precio promedio: -0.348



In [17]:
reto5 = df.groupby('PRODUCTO').agg(
    transacciones=('NRO DOCUMENTO', 'count'),
    volumen_total_kg=('PESO_KG', 'sum'),
    precio_promedio_kg=('USD_KG', 'mean'),
    precio_min_kg=('USD_KG', 'min'),
    precio_max_kg=('USD_KG', 'max'),
    importadores_unicos=('IMPORTADOR', 'nunique')
).round(3).sort_values('transacciones', ascending=False)

print("TABLA 5: Análisis por variante de producto")
print(f"{'='*80}")
display(reto5)

TABLA 5: Análisis por variante de producto


,transacciones,volumen_total_kg,precio_promedio_kg,precio_min_kg,precio_max_kg,importadores_unicos
PRODUCTO,,,,,,
ALULOSA,13,143225,2.892,2.250,3.660,9
ALULOSA EN LIQUIDO,9,213000,2.439,2.350,2.750,2
ALULOSA CRISTALINA,6,240000,2.130,2.130,2.130,1
JARABE DE ALULOSA ALLULOSE SYRUP,4,140000,1.600,1.600,1.600,1
ALULOSA CRISTALINA DOLCIA PRIMA DS C,3,144000,2.680,2.680,2.680,1
ALULOSA EN POLVO ALLULOSE POWDER,2,46800,2.320,2.320,2.320,1
AZUCARES QUIMICAMENTE PUROS ALULOSA,2,9000,2.730,2.730,2.730,1
ALULOSA ALLULOSE,1,5000,2.730,2.730,2.730,1
ALULOSA EN POLVO ALLULOSE PODWER,1,23400,2.320,2.320,2.320,1


## Conclusiones y recomendaciones

### Hallazgos clave

1. **Mercado fragmentado**: 18 importadores participan activamente, pero los 3 principales concentran 42% de transacciones

2. **Dos segmentos de precio**:
   - Compra directa a China: $1.3-$2.0/kg (60% de volumen)
   - Compra vía intermediarios (USA): $2.7-$3.5/kg (40% de volumen)

3. **Crecimiento de abril**: Volumen se triplica pero precio baja, indicando penetración de nuevos compradores con canales directos a China

4. **Independencia precio-volumen**: Ser un importador grande NO garantiza mejores precios; el canal de distribución es clave

### Próximos pasos recomendados

- Monitorear si el crecimiento de abril se sostiene en meses posteriores
- Investigar qué importadores nuevos generaron el salto de volumen en mes 4
- Evaluar si los importadores pueden cambiar de canal (pasar de USA a China) y qué barreras enfrentan

In [18]:
# Exportar tablas resumen
print("Exportando tablas para análisis en BI...")
print()

reto1.to_csv('resumen_importadores.csv')
print("✓ resumen_importadores.csv")

reto2.to_csv('resumen_paises.csv')
print("✓ resumen_paises.csv")

reto3.to_csv('resumen_mensual.csv')
print("✓ resumen_mensual.csv")

reto4.to_csv('volumen_importadores.csv')
print("✓ volumen_importadores.csv")

reto5.to_csv('resumen_productos.csv')
print("✓ resumen_productos.csv")

df.to_csv('dataset_alulosa_procesado.csv', index=False)
print("✓ dataset_alulosa_procesado.csv")

print()
print("Todos los archivos han sido exportados correctamente.")

Exportando tablas para análisis en BI...

✓ resumen_importadores.csv
✓ resumen_paises.csv
✓ resumen_mensual.csv
✓ volumen_importadores.csv
✓ resumen_productos.csv
✓ dataset_alulosa_procesado.csv

Todos los archivos han sido exportados correctamente.
